# 03 - Gold layer: KPI views

Business-ready views on top of the silver star schema. Point your dashboard (Power BI / Databricks Dashboard) at these views.

In [ ]:
%sql
-- Total number of patients
CREATE OR REPLACE VIEW depi.hospitals.total_num_patients_gold AS
SELECT COUNT(DISTINCT patient_sk) AS num_patients
FROM depi.hospitals.fact_table_silver

In [ ]:
%sql
-- Average length of stay (hours) - patients without a valid discharge time are ignored by AVG
CREATE OR REPLACE VIEW depi.hospitals.avg_stay_hours_gold AS
SELECT ROUND(AVG(length_of_stay_hours), 1) AS avg_los_hours
FROM depi.hospitals.fact_table_silver

In [ ]:
%sql
-- Patients per hospital
CREATE OR REPLACE VIEW depi.hospitals.patients_by_hospital_gold AS
SELECT
    h.hospital_name AS hospital,
    COUNT(DISTINCT f.patient_sk) AS num_patients
FROM depi.hospitals.fact_table_silver f
JOIN depi.hospitals.dim_hospital_silver h ON f.hospital_id = h.hospital_id
GROUP BY h.hospital_name
ORDER BY num_patients DESC

In [ ]:
%sql
-- Busiest hospital
CREATE OR REPLACE VIEW depi.hospitals.top_hospital_gold AS
SELECT h.hospital_name AS hospital
FROM depi.hospitals.fact_table_silver f
JOIN depi.hospitals.dim_hospital_silver h ON f.hospital_id = h.hospital_id
GROUP BY h.hospital_name
ORDER BY COUNT(DISTINCT f.patient_sk) DESC
LIMIT 1

In [ ]:
%sql
-- Busiest department
CREATE OR REPLACE VIEW depi.hospitals.top_department_gold AS
SELECT d.department
FROM depi.hospitals.fact_table_silver f
JOIN depi.hospitals.dim_department_silver d ON f.department_sk = d.department_sk
GROUP BY d.department
ORDER BY COUNT(DISTINCT f.patient_sk) DESC
LIMIT 1

In [ ]:
%sql
-- Patients per department   (renamed from df_Num_patients_by_department_gold)
CREATE OR REPLACE VIEW depi.hospitals.patients_by_department_gold AS
SELECT
    d.department,
    COUNT(DISTINCT f.patient_sk) AS num_patients
FROM depi.hospitals.fact_table_silver f
JOIN depi.hospitals.dim_department_silver d ON f.department_sk = d.department_sk
GROUP BY d.department
ORDER BY num_patients DESC

In [ ]:
%sql
-- Average length of stay per department   (renamed from length_of_stay_hours_gold)
CREATE OR REPLACE VIEW depi.hospitals.avg_los_by_department_gold AS
SELECT
    d.department,
    ROUND(AVG(f.length_of_stay_hours), 1) AS avg_los_hours
FROM depi.hospitals.fact_table_silver f
JOIN depi.hospitals.dim_department_silver d ON f.department_sk = d.department_sk
GROUP BY d.department
ORDER BY avg_los_hours DESC

In [ ]:
%sql
-- Patients per gender
CREATE OR REPLACE VIEW depi.hospitals.num_patients_by_gender_gold AS
SELECT
    p.gender,
    COUNT(DISTINCT f.patient_sk) AS num_patients
FROM depi.hospitals.fact_table_silver f
JOIN depi.hospitals.dim_patient_silver p ON f.patient_sk = p.patient_sk
GROUP BY p.gender
ORDER BY num_patients DESC

In [ ]:
%sql
-- Patients per age (invalid ages are NULL in silver, so they are excluded here)
CREATE OR REPLACE VIEW depi.hospitals.num_patients_by_age_gold AS
SELECT
    p.age,
    COUNT(DISTINCT f.patient_sk) AS num_patients
FROM depi.hospitals.fact_table_silver f
JOIN depi.hospitals.dim_patient_silver p ON f.patient_sk = p.patient_sk
WHERE p.age IS NOT NULL
GROUP BY p.age
ORDER BY p.age

In [ ]:
%sql
-- Patients per age group
CREATE OR REPLACE VIEW depi.hospitals.num_patients_by_age_group_gold AS
SELECT
    p.age_group,
    COUNT(DISTINCT f.patient_sk) AS num_patients
FROM depi.hospitals.fact_table_silver f
JOIN depi.hospitals.dim_patient_silver p ON f.patient_sk = p.patient_sk
GROUP BY p.age_group
ORDER BY p.age_group

In [ ]:
%sql
-- Admissions per day
CREATE OR REPLACE VIEW depi.hospitals.daily_patients_gold AS
SELECT
    d.date_sk,
    d.date,
    COUNT(DISTINCT f.patient_sk) AS num_admissions
FROM depi.hospitals.fact_table_silver f
JOIN depi.hospitals.dim_date_silver d ON f.admission_date_sk = d.date_sk
GROUP BY d.date_sk, d.date
ORDER BY d.date_sk

In [ ]:
%sql
-- Admissions per hour of the day (peak hours)
CREATE OR REPLACE VIEW depi.hospitals.admissions_by_hour_gold AS
SELECT
    HOUR(admission_time) AS hour_of_day,
    COUNT(DISTINCT patient_sk) AS num_admissions
FROM depi.hospitals.fact_table_silver
GROUP BY HOUR(admission_time)
ORDER BY hour_of_day

In [ ]:
%sql
-- Patients received in the last hour  (read by the alert notebook)
CREATE OR REPLACE VIEW depi.hospitals.patients_last_hour_gold AS
SELECT COUNT(DISTINCT patient_sk) AS num_patients_last_hour
FROM depi.hospitals.fact_table_silver
WHERE event_ingestion_time >= current_timestamp() - INTERVAL 1 HOUR

In [ ]:
%sql
-- Clean up views that were renamed / replaced
DROP VIEW IF EXISTS depi.hospitals.df_Num_patients_by_department_gold;
DROP VIEW IF EXISTS depi.hospitals.length_of_stay_hours_gold

In [ ]:
# Sanity check: every gold view should be queryable
views = [r.viewName for r in spark.sql("SHOW VIEWS IN depi.hospitals").collect() if r.viewName.endswith("_gold")]
for v in sorted(views):
    print(f"{v:40s} rows = {spark.table('depi.hospitals.' + v).count()}")